## 环境

```shell
conda create -n SpatialScores python=3.10 -y
# 安装cu128,兼容flash-attn
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
# 可能指定transformers,兼容llava-ov
pip install transformers
```

# 评估模型

In [1]:
import os
import json
import torch
import argparse
import transformers
from tqdm import tqdm
from PIL import Image
import gc
from tool import process_model_input
from qwen_vl_utils import process_vision_info

from transformers import AutoProcessor, AutoTokenizer, AutoModel, AutoModelForCausalLM
if transformers.__version__ >= "4.49": # Note: cambrian needs transformers == 4.45.0, while others need >= 4.49.0
    from transformers import LlavaForConditionalGeneration, LlavaOnevisionForConditionalGeneration, Qwen2VLForConditionalGeneration, Qwen2_5_VLForConditionalGeneration, AutoModelForImageTextToText


IMAGE_TOKEN_INDEX = -200

from utils.util import image_to_base64_data_uri, extract_number, extract_yes_no, extract_option, extract_numeric_with_unit, load_image

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

# Optimize GPU memory allocation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
SEED = 42
torch.manual_seed(SEED)

/home/alvis/miniconda3/envs/llava/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# MODULE: 加载模型及相关组件
def load_model_and_components(model_path, model_name="llava-ov-7b"):
    # 清理缓存
    global_vars = globals()
    for var_name in ['model', 'tokenizer_or_processor', 'image_processor']:
        if var_name in global_vars:
            del global_vars[var_name]
    gc.collect()
    torch.cuda.empty_cache()
    image_processor = None
    if model_name in ["llava-ov-1.5-8b"]:
        model = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype="auto", device_map="auto", trust_remote_code=True,
        )
        tokenizer_or_processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    model.eval()
    return model, tokenizer_or_processor, image_processor

In [3]:
def generate_response(model, tokenizer_or_processor, model_inputs, model_name="qwen2_5vl-7b", model_config=None, image_processor=None):
    NUM_BEAMS, TEMPERATURE, MAX_NEW_TOKENS, USE_CACHE = 1, 0.0, 1024, True
    DO_SAMPLE = True if TEMPERATURE > 0 else False
    MAX_NEW_TOKENS = 2048 if model_name == 'kimivl-3b-thinking' else MAX_NEW_TOKENS
    
    device = next(model.parameters()).device
    assistant_prompt, image_paths, images, text = model_inputs['assistant_prompt'], model_inputs['img_paths'], model_inputs['images'], model_inputs['question']

    with torch.no_grad():
        if model_name in ["llava-ov-1.5-8b"]:
            for i in range(len(images)):
                if images[i].size[0] <= 3 or images[i].size[1] <= 3:
                    images[i] = images[i].resize((32, 32), Image.BICUBIC)
            # 2. 构造符合LLaVA-OneVision要求的messages（适配多图像输入，对齐用户prompt）
            user_content = []
            # 遍历本地图像列表，添加每个图像的信息（type=image，image=PIL图像对象）
            for img in images: # 此处传入本地PIL图像，替代官方示例中的网络图片URL
                user_content.append({
                    "type": "image",
                    "image": img  
                })
            user_content.append({ # 添加文本prompt（和前面分支的text保持一致）
                "type": "text",
                "text": text
            })
            messages = [ # 构造对话消息：注意llava-ov-1.5-8b的对话格式（无需提前添加assistant历史，add_generation_prompt=True会自动添加）
                {
                    "role": "user",
                    "content": user_content  # 包含所有图像 + 文本提问
                }
            ]
            # 3. 复用官方推理流程：处理文本模板和视觉输入
            # 生成对话模板文本（不进行分词，添加生成提示）
            input_text = tokenizer_or_processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True  # 自动添加"assistant:"生成前缀
            )

            # 处理视觉信息（分离图像和视频输入，此处无视频，video_inputs为空）
            image_inputs, video_inputs = process_vision_info(messages)

            # 4. 构造模型输入：批量处理（适配单样本/多样本，和前面分支逻辑对齐）
            inputs = tokenizer_or_processor(
                text=[input_text],  # 文本输入封装为列表（批量格式）
                images=image_inputs,  # 预处理后的图像输入
                videos=video_inputs,  # 无视频则传入空列表
                padding=True,  # 自动填充
                return_tensors="pt"  # 返回PyTorch张量
            ).to(device)  # 移至目标设备（cuda/cpu，和前面分支的device保持一致）

            # 5. 模型生成：使用和前面分支相同的生成参数
            generated_ids = model.generate(
                **inputs,
                num_beams=NUM_BEAMS,
                temperature=TEMPERATURE,
                max_new_tokens=MAX_NEW_TOKENS,
                use_cache=USE_CACHE,
                do_sample=DO_SAMPLE
            )

            # 6. 裁剪生成结果：去除输入部分，只保留模型新增生成内容（关键步骤，避免输出包含输入prompt）
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            # 7. 解码生成结果：和前面分支的解码参数保持一致
            output_text_list = tokenizer_or_processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )

            # 8. 提取响应文本：对齐前面分支的response格式（处理单样本场景，取列表第一个元素）
            response = output_text_list[0].strip()

            # 可选：和前面分支保持一致，进一步清理响应文本（若存在多余的"assistant"标识）
            if "assistant" in response:
                response = response.split("assistant")[-1].strip()
            if "ASSISTANT" in response:
                response = response.split("ASSISTANT")[-1].strip()

    torch.cuda.empty_cache()
    
    return response

## 小样本测试

In [4]:
model_path = "./models/LLaVA-OneVision-1.5-8B-Instruct"
model_name = "llava-ov-1.5-8b"

# TEST: 尝试模型能否加载
# from transformers import logging
# logging.set_verbosity_info()
model, tokenizer_or_processor, image_processor = load_model_and_components(
    model_path=model_path,
    model_name=model_name,
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.45it/s]


In [5]:
# 从dataset/SpatialScore_subset.json随机选择5个样本进行测试
import random
with open("./dataset/SpatialScore_subset.json", 'r') as f:
    data = json.load(f)
sampled_data = random.sample(data, 5)
from pprint import pprint
for sample in sampled_data:
    model_inputs = process_model_input(sample, model_name=model_name)
    pprint(model_inputs)
    results = generate_response(model, tokenizer_or_processor, model_inputs, model_name=model_name, model_config=getattr(model, 'config', None), image_processor=image_processor)
    print("Question:", sample['question'])
    print("Predicted Answer:", extract_option(results))
    print("Ground Truth Answer:", extract_option(sample['answer']))
    print("-" * 50)

{'answer': 'A',
 'assistant_prompt': '**Please select the most appropriate answer from options '
                     '(A), (B), (C), (D), (E), or (F).**\n'
                     '**Respond ONLY with the letter and its parentheses, for '
                     'example: (A)**\n'
                     '\n'
                     'Question: ',
 'category': '3D Positional Relation',
 'category_origin': 'orientation_on_the_left',
 'images': [<PIL.Image.Image image mode=RGB size=640x640 at 0x71762DCAB7F0>],
 'img_paths': ['./dataset/3DSRBench/coco_images/train2017/000000053058.jpg'],
 'index': 3189,
 'index_origin': '1INS9OIW-flip',
 'input_modality': 'single-image',
 'question': 'Consider the real-world 3D locations and orientations of the '
             "objects. If I stand at the person's position facing where it is "
             'facing, is the printer on the left or right of me?\n'
             '(A) on the left\n'
             '(B) on the right',
 'question_type': 'multi-choice',
 'source':

## 大测试

In [6]:
from evaluate import run_test
run_test(
    model = model,
    tokenizer_or_processor = tokenizer_or_processor,
    image_processor = image_processor,
    model_path=model_path,
    model_name=model_name,
    generate_response=generate_response,
)

Loaded 4000 items from unified benchmark


Processing SpatialScore items: 100%|██████████| 4000/4000 [1:04:57<00:00,  1.03it/s]


Source: MMVP - Accuracy: 68.67% (206/300, score: 206.00)
Source: VSI-Bench_8 - Accuracy: 26.00% (52/200, score: 52.00)
Source: 3DSRBench - Accuracy: 56.87% (853/1500, score: 853.00)
Source: cvbench - Accuracy: 65.90% (659/1000, score: 659.00)
Source: MMIU - Accuracy: 24.44% (220/900, score: 220.00)
Source: BLINK - Accuracy: 42.00% (42/100, score: 42.00)
Category: Others - Accuracy: 51.60% (258/500, score: 258.00)
Category: Object Properties - Accuracy: 51.40% (257/500, score: 257.00)
Category: Object Localization - Accuracy: 62.40% (312/500, score: 312.00)
Category: 3D Positional Relation - Accuracy: 56.80% (284/500, score: 284.00)
Category: Counting - Accuracy: 64.20% (321/500, score: 321.00)
Category: Depth and Distance - Accuracy: 67.60% (338/500, score: 338.00)
Category: Point and Object Tracking - Accuracy: 21.60% (108/500, score: 108.00)
Category: Camera and Image Transformation - Accuracy: 30.80% (154/500, score: 154.00)
Overall Accuracy: 50.80% (2032/4000, score: 2032.00)
All r

```shell
Loaded 4000 items from unified benchmark
Processing SpatialScore items: 100%|██████████| 4000/4000 [1:04:57<00:00,  1.03it/s]
Source: MMVP - Accuracy: 68.67% (206/300, score: 206.00)
Source: VSI-Bench_8 - Accuracy: 26.00% (52/200, score: 52.00)
Source: 3DSRBench - Accuracy: 56.87% (853/1500, score: 853.00)
Source: cvbench - Accuracy: 65.90% (659/1000, score: 659.00)
Source: MMIU - Accuracy: 24.44% (220/900, score: 220.00)
Source: BLINK - Accuracy: 42.00% (42/100, score: 42.00)
Category: Others - Accuracy: 51.60% (258/500, score: 258.00)
Category: Object Properties - Accuracy: 51.40% (257/500, score: 257.00)
Category: Object Localization - Accuracy: 62.40% (312/500, score: 312.00)
Category: 3D Positional Relation - Accuracy: 56.80% (284/500, score: 284.00)
Category: Counting - Accuracy: 64.20% (321/500, score: 321.00)
Category: Depth and Distance - Accuracy: 67.60% (338/500, score: 338.00)
Category: Point and Object Tracking - Accuracy: 21.60% (108/500, score: 108.00)
Category: Camera and Image Transformation - Accuracy: 30.80% (154/500, score: 154.00)
Overall Accuracy: 50.80% (2032/4000, score: 2032.00)
All results saved to ./eval_results/llava-ov-1.5-8b
```